# Magnetic Steering of SPION Suspensions at a Bifurcation — Data Analysis

This notebook examines simultaneous optical recordings upstream of the bifurcation and in branches A and B during E02. This first stage imports the measurements, establishes a shared time axis, and displays the raw signals and event markers.

Injection-rate units and the concentrations of the test and reference suspensions require confirmation before quantitative analysis.

## Setup

`project_dir` locates the repository; `data_dir` contains the original measurements. The selected wavelength is $\lambda = 640\,\mathrm{nm}$. Imports and analysis parameters are defined here.

In [ ]:
from pathlib import Path
import sys

import pandas as pd
import matplotlib.pyplot as plt

project_dir = Path("/Users/lukaskroll/Documents/10_Projekte/dynano/sensor_modell")
if str(project_dir) not in sys.path:
    sys.path.insert(0, str(project_dir))

from src.spectral_sensor.io import read_sensor_csv, read_marker_rows, read_protocol
from src.spectral_sensor.preprocessing import synchronize_time
from src.spectral_sensor.plotting import plot_measurements, save_plot

experiment_dir = project_dir / "experiments/E02_20x15_mag_steering"
data_dir = experiment_dir / "data"
results_dir = experiment_dir / "results"
wavelength_nm = 640

sensor_styles = {
    "sensor_0": {"color": "#004488"},
    "sensor_1": {"color": "#DDAA33"},
    "sensor_2": {"color": "#BB5566"},
}

## Measurement metadata

- Experiment: E02; batch: XL004.
- Operator: Lukas Kroll; location: Laboratory A701, FH Dortmund.
- Protocol start: 2026-07-08T13:01:43.977864; filename timestamp: 20260708_113243. These source timestamps are retained separately. The May date in the earlier report is inconsistent with these files.
- Sensor settings in `Protocol.md`: ATIME 0, ASTEP 8993, GAIN 1, IRMAIN 25, IROPP 25, VISM 255, VISLR 255.
- Physical sensor IDs and system roles below follow `E02_Evaluation.md`; software IDs are checked against the measurement files.

`read_protocol` extracts the supported protocol fields. `sensor_metadata` preserves the distinction between physical identity, recorded software ID, and system role.

In [ ]:
sensor_metadata = pd.DataFrame([
    {"sensor": "sensor_0", "software_id": 1002, "role": "Upstream reference", "file": "SpectralSensor 3_20260708_113243.csv"},
    {"sensor": "sensor_1", "software_id": 1001, "role": "Branch A", "file": "SpectralSensor 2_20260708_113243.csv"},
    {"sensor": "sensor_2", "software_id": 1000, "role": "Branch B", "file": "SpectralSensor 1_20260708_113243.csv"},
]).set_index("sensor")

protocol_metadata = read_protocol(data_dir / "Protocol.md")
display(pd.DataFrame.from_dict(protocol_metadata, orient="index", columns=["value"]))
display(sensor_metadata)

## Data import and synchronization

`read_sensor_csv` and `read_marker_rows` load the original tables. `synchronize_time` uses one shared origin $t_0$ for all recordings, with $t = (t_{\mathrm{raw}} - t_0)/1000$ in seconds. It does not correct independent clock offsets. The original timestamps remain available.

In [ ]:
sensor_data = {
    sensor: read_sensor_csv(data_dir / row["file"])
    for sensor, row in sensor_metadata.iterrows()
}
markers = read_marker_rows(data_dir / "Markers_20260708_113243.csv")

for sensor, row in sensor_metadata.iterrows():
    assert sensor_data[sensor]["sensor"].eq(row["software_id"]).all(), sensor

synced_sensors, synced_markers, start_ms = synchronize_time(sensor_data, markers)
display(synced_markers)

## Raw signals

`plot_measurements` displays the ADC signal $I(t)$ and recorded event times. Sensor colors retain the physical identities used in E01. These plots provide the initial check of timing, signal range, and injection responses.

In [ ]:
for sensor, row in sensor_metadata.iterrows():
    fig, ax = plot_measurements(
        {sensor: synced_sensors[sensor]},
        synced_markers,
        time_column="time_s",
        signal_column=str(wavelength_nm),
        xlabel="Time since measurement start [s]",
        ylabel=f"Signal at {wavelength_nm} nm [ADC counts]",
        title=f"{sensor} — {row["role"]}",
        sensor_styles=sensor_styles,
    )
    plt.show()

## Next analysis step

The recorded sequence contains one reference injection at STEP 6 and twelve test injections: STEP 8–10 (50 µL, magnet off), STEP 13–15 (50 µL, magnet on), STEP 16–18 (100 µL, magnet on), and STEP 19–21 (150 µL, magnet on). STEP identifiers differ from the DataFrame row index.

After confirming the injection rate and suspension concentrations, define the baseline and integration windows explicitly. The existing concentration curves and AES comparison are prior results; they are not reproduced by this initial notebook.